# NB09-batch — SAM3 prompt batching: correctness + speed experiment

**Runs on Kaggle T4 GPU.** Isolated from the NB09 sweep kernels — this notebook
does not render tiles, run sweeps, or write to any NB09 output folder.

Question: NB09's `collect_class_scores` asks SAM3 about each class **one at a
time** — 1 shared image-encoder pass, then N sequential grounding-transformer
forwards. SAM3's `FindStage` supports `img_ids=[0]*N` / `text_ids=arange(N)`,
which gathers the same crop features N times and runs the encoder+decoder once
over a batch of N. Does that give identical logits, and is it faster?

Measured here: per-class max|diff| between the two paths, argmax agreement,
wall-clock mean/std over multiple crops x reps, and peak GPU memory for each.


## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/tmp/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX \u2192 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/tmp/SegEarth-OV-3")
# nb09-batch-experiment lives on this branch, not on master (the default).
BRANCH = "nb09-overnight-iteration"

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated \u2192 {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1", "-b", BRANCH,
         "https://github.com/HarishDeepak/rg-segearth-ov3", str(REPO)],
        check=True)
    print(f"Cloned \u2192 {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — Run the comparison

`batch_prompts_experiment.py` lives in the cloned repo at
`notebooks/push/nb09-batch-experiment/`. The baseline path is NB09 cell 11's
`collect_class_scores` copied verbatim, so "sequential" here is exactly what the
sweeps run today.


In [ ]:
%%bash
export PYTHONUNBUFFERED=1
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3
python notebooks/push/nb09-batch-experiment/batch_prompts_experiment.py \
    --tile dop20_32_476_5524_1_he --crop 768 --crops 4 --reps 3


## 4 — VRAM headroom: how large can N get?

The batched call holds N copies of the decoder/segmentation activations at once.
N=9 is this tile's real class count; this sweeps N upward on one crop to find
where a T4's 16GB actually stops us, since synonym-expanded prompt files
(`cls_hessen.txt`) can reach 20+ queries.


In [ ]:
%%bash
export PYTHONUNBUFFERED=1
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3
python - << 'PYEOF'
import sys, torch
from pathlib import Path
sys.path.insert(0, "notebooks/push/nb09-batch-experiment")
from batch_prompts_experiment import (WORDS, cache_text_batched,
                                      collect_class_scores_batched, crops_from_tile)
from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

img = sorted(Path("/kaggle/input").rglob("dop20_32_476_5524_1_he.jpg"))[0]
model = build_sam3_image_model(bpe_path="./sam3/assets/bpe_simple_vocab_16e6.txt.gz",
                               checkpoint_path=SAM3_CHECKPOINT, device="cuda")
model.eval()
for p in model.parameters(): p.requires_grad = False
proc = Sam3Processor(model, confidence_threshold=0.1, device="cuda")
crop = crops_from_tile(img, 768, 1)[0]
w, h = crop.size

for N in [6, 9, 12, 16, 20, 24, 32]:
    words = (WORDS * 8)[:N]
    try:
        te = cache_text_batched(model, words)
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
            st = proc.set_image(crop)
            collect_class_scores_batched(model, st, h, w, te, N, "cuda", 0.1)
        print(f"N={N:3d}  OK   peak {torch.cuda.max_memory_allocated()/1e9:.2f} GB", flush=True)
    except torch.cuda.OutOfMemoryError:
        print(f"N={N:3d}  OOM", flush=True)
        torch.cuda.empty_cache()
        break
PYEOF
